# TP4 — Detección de eventos atípicos en GDELT 2.0

**Curso:** Introducción a la Ingeniería de Datos — Doctorado
**Autor:** Marcos Kantor

En el TP1 procesamos el flujo de GDELT 2.0, en el TP2 diseñamos un almacén de datos para el negocio periodístico y en el TP3 analizamos el comportamiento multivariante del ecosistema global de noticias. En este TP buscamos **eventos "raros"** que valga la pena investigar.

**Pipeline del notebook**

| Etapa | Qué se hace | Producto |
|---|---|---|
| 1. Ingesta | Descarga de las tablas *Event* (`export.CSV.zip`) de 15 minutos de los últimos 7 días desde `masterfilelist.txt`, verificación MD5 y desnormalización con los libros de códigos CAMEO/FIPS | `gdelt_obt_7d.parquet` (One Big Table) |
| 2.1 Nulos y ceros | Diagnóstico de faltantes y ceros, criterio de descarte/imputación por variable | matriz analítica limpia |
| 2.2 Cardinalidad | Categorías con frecuencia < 5 % → `OTHER` | categóricas reducidas |
| 2.3 Variables indicadoras | *One-hot encoding* (dummies) con categoría de referencia | matriz numérica |
| 2.4 ACP | Escalado + Análisis por Componentes Principales, selección del número mínimo de componentes | matriz de *scores* |
| 2.5 Atípicos | 10 métodos de 6 familias (distancia global, ACP T²/SPE, K-Means, k-NN, LOF/DBSCAN, Isolation Forest, autoencoder), comparación por Jaccard y consenso por mayoría | `gdelt_atipicos.parquet` / `.csv` |

## Declaración de uso de Inteligencia Artificial

| Herramienta | Propósito |
|---|---|
| **Claude Opus 5.5** (Anthropic, vía Claude Code) | Generación del código fuente Python del notebook (ingesta, limpieza, ACP y detección de atípicos) y borrador de las celdas Markdown explicativas. |

Todas las decisiones metodológicas (umbral del 5 % de cardinalidad, criterio de imputación, regla de selección de componentes, nivel de significancia de los límites de control y regla de consenso) fueron definidas, revisadas y validadas por el autor. Los resultados numéricos se obtienen ejecutando el notebook; no fueron generados por la IA.

### Guía de instrucciones para reproducir el resultado (*Human-in-the-loop*)

La siguiente secuencia de instrucciones permite reproducir el notebook con la herramienta de IA. Cada paso indica la instrucción dada y la **verificación humana** realizada antes de pasar al siguiente.

1. **Contexto y arquitectura.**
   *"Actuá como ingeniero de datos. Necesito un notebook de Google Colab en Python que construya un pipeline batch sobre la tabla de hechos Event de GDELT 2.0 (61 campos, codebook V2.0). El pipeline tiene etapas: ingesta → One Big Table desnormalizada → limpieza → reducción de cardinalidad → variables indicadoras → ACP → detección de atípicos multivariantes. Usá pandas, pyarrow, scikit-learn y scipy; todo parametrizado en una celda de configuración."*
   ✔ Verifiqué que los 61 nombres de columna coincidan con el codebook GDELT-Event_Codebook-V2.0 y que los códigos CAMEO (`EventCode`, `EventRootCode`) se lean como texto para no perder ceros a la izquierda.

2. **Ingesta incremental e idempotente.**
   *"Leé `masterfilelist.txt` en streaming, filtrá solo los archivos `.export.CSV.zip` cuya marca temporal (YYYYMMDDHHMMSS) esté dentro de una ventana deslizante de 7 días en UTC, descargalos en paralelo con reintentos, validá el MD5 y el tamaño informados en la lista maestra, cacheá en disco y registrá los intervalos de 15 minutos faltantes."*
   ✔ Controlé la completitud: 7 × 96 = 672 intervalos esperados vs. descargados.

3. **Desnormalización (One Big Table).**
   *"Construí una OBT: a la tabla de hechos agregale las descripciones de las dimensiones (lookups CAMEO de eventos, países, tipos de actor, grupos, religiones, etnias; FIPS para geografía; QuadClass y Geo_Type), atributos derivados de tiempo (fecha de publicación, hora, día de semana, días de retraso entre SQLDATE y DATEADDED) y el dominio de la fuente. Eliminá duplicados por GLOBALEVENTID y persistí en Parquet con compresión snappy."*
   ✔ Revisé que el *grain* sea un evento por fila y que no se multipliquen filas al desnormalizar (uso de `map` en lugar de `merge`).

4. **Calidad de datos: nulos y ceros.**
   *"Generá un perfil de faltantes con % de nulos y % de ceros por columna. Distinguí ceros estructurales (GoldsteinScale y AvgTone = 0 son valores válidos, neutros) de ceros que representan faltantes (lat = long = 0, 'Null Island'). Para categóricas, tratá el nulo como categoría 'UNKNOWN' porque es faltante no aleatorio (MNAR); imputá coordenadas por mediana del país del ActionGeo; imputá GoldsteinScale por la moda del EventCode, ya que es función determinística del código CAMEO; descartá el resto y reportá cuántas filas se pierden."*
   ✔ Validé que el descarte de filas sea marginal (< 5 %) antes de aceptar el criterio.

5. **Transformaciones.**
   *"Aplicá log1p a las variables de conteo con asimetría fuerte (NumMentions, NumSources, NumArticles, DiasRetraso) y convertí lat/long a coordenadas cartesianas sobre la esfera unitaria para evitar la discontinuidad en ±180°. Mostrá la asimetría antes y después."*

6. **Reducción de cardinalidad y variables indicadoras.**
   *"Para cada categórica agrupá en 'OTHER' las categorías con frecuencia relativa menor al 5 % de la muestra; mostrá cardinalidad antes/después. Luego hacé one-hot encoding eliminando una categoría de referencia ('OTHER' si existe, si no la más frecuente) para evitar la colinealidad perfecta (trampa de las dummies)."*

7. **ACP.**
   *"Estandarizá (z-score) la matriz, ajustá PCA con todos los componentes, graficá scree plot y varianza acumulada, y compará criterios: varianza acumulada ≥ 80 % y ≥ 90 %, Kaiser (autovalor > 1) y codo. Elegí el número mínimo de componentes con el criterio de 80 % de varianza explicada y mostrá las cargas (loadings)."*
   ✔ Revisé que la cantidad elegida sea coherente entre criterios e interpreté las primeras componentes.

8. **Atípicos multivariantes (siguiendo el notebook de referencia de la cátedra *Cardiotocography outliers*).**
   *"Tomá como guía las familias de métodos del notebook de Cardiotocography y aplicalas sobre los scores del ACP: distancia euclídea al centroide con Z-score > 3 (verificando el supuesto de normalidad con curtosis y Anderson-Darling) y Tukey fences 2·IQR; T² de Hotelling con límite χ²(0,999; K) y SPE con la aproximación de Box; K-Means (distancia al centroide del grupo), k-NN (distancia media a los vecinos), LOF y DBSCAN (eps por gráfico de k-distancias); Isolation Forest y un autoencoder denso en PyTorch con error de reconstrucción. Como GDELT no tiene etiquetas, reemplazá la matriz de confusión por el índice de Jaccard entre métodos y un consenso por mayoría de votos. Para escalar a ~10⁶ eventos, ajustá los métodos O(n²) sobre una muestra de referencia y evaluá todos los eventos en modo novelty. Explicá cada atípico con las variables de mayor z-score y residuo y exportalo a Parquet y CSV con todos los atributos de la OBT."*
   ✔ Inspeccioné manualmente los 25 atípicos más extremos abriendo su `SOURCEURL`.

## 0. Configuración del entorno

In [ ]:
# En Colab todas estas librerías ya están instaladas; se fuerza pyarrow por las dudas.
!pip -q install pyarrow

In [ ]:
import csv, hashlib, io, re, time, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.stats import chi2, zscore, anderson
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ----------------------------- parámetros del pipeline -----------------------------
DIAS          = 7          # ventana deslizante de ingesta
MAX_ARCHIVOS  = None       # p.ej. 96 para una prueba rápida (1 día); None = ventana completa
MAX_WORKERS   = 16         # descargas en paralelo
UMBRAL_RARO   = 0.05       # categorías con frecuencia < 5 % -> 'OTHER'
UMBRAL_VAR    = 0.80       # varianza acumulada mínima para elegir K componentes
ALPHA         = 0.999      # nivel de confianza de los límites de control (T², SPE)
TOP_FRAC      = 0.01       # fracción superior marcada por los métodos de score continuo
N_REF         = 20_000     # muestra de referencia para k-NN / LOF / DBSCAN (métodos O(n²))
K_CLUSTERS    = 10         # grupos de K-Means
K_VECINOS     = 10         # vecinos para k-NN y LOF
DIM_LATENTE   = 4          # dimensión del espacio latente del autoencoder
EPOCAS_AE     = 15         # épocas de entrenamiento del autoencoder
SEED          = 42

DATA_DIR = Path("/content/gdelt_tp4") if Path("/content").exists() else Path("gdelt_tp4")
RAW_DIR  = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OBT_PATH = DATA_DIR / "gdelt_obt_7d.parquet"
ATI_PATH = DATA_DIR / "gdelt_atipicos.parquet"
print("Directorio de trabajo:", DATA_DIR.resolve())

## 1. Ingesta de datos

### 1.1 Esquema de la tabla *Event* (GDELT 2.0)

Cada archivo `YYYYMMDDHHMMSS.export.CSV.zip` contiene un TSV sin encabezado con **61 campos** (codebook V2.0). Los códigos CAMEO y de país se leen como **texto** para conservar ceros a la izquierda (`"010"` ≠ `10`).

In [ ]:
ACTOR = ["Code", "Name", "CountryCode", "KnownGroupCode", "EthnicCode",
         "Religion1Code", "Religion2Code", "Type1Code", "Type2Code", "Type3Code"]
GEO   = ["Type", "FullName", "CountryCode", "ADM1Code", "ADM2Code", "Lat", "Long", "FeatureID"]

COLS = (["GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate"]
        + [f"Actor1{c}" for c in ACTOR] + [f"Actor2{c}" for c in ACTOR]
        + ["IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode", "QuadClass",
           "GoldsteinScale", "NumMentions", "NumSources", "NumArticles", "AvgTone"]
        + [f"Actor1Geo_{c}" for c in GEO] + [f"Actor2Geo_{c}" for c in GEO]
        + [f"ActionGeo_{c}" for c in GEO]
        + ["DATEADDED", "SOURCEURL"])
assert len(COLS) == 61

COLS_NUM = (["IsRootEvent", "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
             "NumArticles", "AvgTone", "Year", "FractionDate"]
            + [f"{p}Geo_{c}" for p in ("Actor1", "Actor2", "Action") for c in ("Type", "Lat", "Long")])

### 1.2 Selección de archivos desde `masterfilelist.txt`

La lista maestra tiene una línea por archivo: `tamaño md5 url`. Se lee en *streaming* (es grande) y se filtran sólo las tablas de evento (`export.CSV.zip`) dentro de la ventana de 7 días en UTC.

In [ ]:
MASTER_URL = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"

def listar_exports(dias: int) -> pd.DataFrame:
    corte = (datetime.now(timezone.utc) - timedelta(days=dias)).strftime("%Y%m%d%H%M%S")
    filas = []
    with requests.get(MASTER_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        for linea in r.iter_lines(decode_unicode=True):
            if not linea or not linea.endswith(".export.CSV.zip"):
                continue
            partes = linea.split()
            if len(partes) != 3:
                continue
            size, md5, url = partes
            ts = url.rsplit("/", 1)[-1].split(".")[0]
            if ts >= corte:
                filas.append((ts, int(size), md5, url))
    return (pd.DataFrame(filas, columns=["timestamp", "size", "md5", "url"])
              .drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True))

archivos = listar_exports(DIAS)
if MAX_ARCHIVOS:
    archivos = archivos.tail(MAX_ARCHIVOS).reset_index(drop=True)

esperados = DIAS * 96 if not MAX_ARCHIVOS else MAX_ARCHIVOS
print(f"Archivos en ventana: {len(archivos)} (esperados ≈ {esperados})")
print("Desde", archivos.timestamp.min(), "hasta", archivos.timestamp.max())
archivos.head()

### 1.3 Descarga paralela con verificación de integridad

Cada archivo se descarga con reintentos, se valida contra el **MD5** de la lista maestra y se cachea en disco (re-ejecutar el notebook no vuelve a descargar). El parseo convierte tipos por archivo para controlar el uso de memoria.

In [ ]:
def descargar(row) -> Path:
    destino = RAW_DIR / row.url.rsplit("/", 1)[-1]
    if destino.exists() and destino.stat().st_size == row.size:
        return destino
    err = None
    for intento in range(4):
        try:
            r = requests.get(row.url, timeout=90)
            r.raise_for_status()
            if hashlib.md5(r.content).hexdigest() != row.md5:
                raise ValueError("MD5 no coincide")
            destino.write_bytes(r.content)
            return destino
        except Exception as e:
            err = e
            time.sleep(2 * (intento + 1))
    raise RuntimeError(f"{row.url}: {err}")

def leer_export(path: Path, ts: str) -> pd.DataFrame:
    with zipfile.ZipFile(path) as z, z.open(z.namelist()[0]) as f:
        df = pd.read_csv(f, sep="\t", header=None, names=COLS, dtype=str,
                         quoting=csv.QUOTE_NONE, on_bad_lines="skip",
                         keep_default_na=False, na_values=[""],
                         encoding="utf-8", encoding_errors="replace")
    for c in COLS_NUM:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["GLOBALEVENTID"] = pd.to_numeric(df["GLOBALEVENTID"], errors="coerce")
    df["ArchivoOrigen"] = ts
    return df

def procesar(row):
    return leer_export(descargar(row), row.timestamp)

partes, fallidos = [], []
with ThreadPoolExecutor(MAX_WORKERS) as ex:
    futuros = {ex.submit(procesar, row): row.timestamp for row in archivos.itertuples()}
    for fut in tqdm(as_completed(futuros), total=len(futuros), desc="Event 15'"):
        try:
            partes.append(fut.result())
        except Exception as e:
            fallidos.append((futuros[fut], str(e)))

eventos = pd.concat(partes, ignore_index=True)
del partes
print(f"Intervalos OK: {len(archivos) - len(fallidos)} | fallidos: {len(fallidos)}")
print(f"Filas leídas: {len(eventos):,}")

### 1.4 Construcción de la One Big Table (OBT)

La OBT **desnormaliza** el modelo estrella del TP2: a la tabla de hechos se le agregan los atributos descriptivos de las dimensiones (libros de códigos CAMEO / FIPS publicados por GDELT) y atributos derivados de tiempo y fuente. Se usa `Series.map` con diccionarios para garantizar que el *grain* (un evento por fila) no cambie.

In [ ]:
LOOKUP_URL = "https://www.gdeltproject.org/data/lookups/{}"

def cargar_lookup(nombre: str) -> dict:
    try:
        lk = pd.read_csv(LOOKUP_URL.format(nombre), sep="\t", header=None, usecols=[0, 1],
                         names=["code", "label"], dtype=str, quoting=csv.QUOTE_NONE,
                         encoding_errors="replace")
        lk = lk.dropna()
        lk = lk[~lk["code"].str.upper().str.contains("CODE")]          # descarta encabezado
        return dict(zip(lk["code"].str.strip(), lk["label"].str.strip()))
    except Exception as e:
        print(f"[aviso] no se pudo cargar {nombre}: {e}")
        return {}

LK = {n: cargar_lookup(f"{n}.txt") for n in
      ["CAMEO.eventcodes", "CAMEO.country", "CAMEO.type", "CAMEO.knowngroup",
       "CAMEO.ethnic", "CAMEO.religion", "FIPS.country"]}
print({k: len(v) for k, v in LK.items()})

QUAD = {1: "Verbal Cooperation", 2: "Material Cooperation", 3: "Verbal Conflict", 4: "Material Conflict"}
GEOT = {1: "COUNTRY", 2: "USSTATE", 3: "USCITY", 4: "WORLDCITY", 5: "WORLDSTATE"}

In [ ]:
obt = eventos.drop_duplicates("GLOBALEVENTID").copy()
del eventos
print("Filas tras eliminar duplicados por GLOBALEVENTID:", f"{len(obt):,}")

# --- dimensión Evento (CAMEO)
for c in ["EventCode", "EventBaseCode", "EventRootCode"]:
    obt[c.replace("Code", "Desc")] = obt[c].map(LK["CAMEO.eventcodes"])
obt["QuadClassDesc"] = obt["QuadClass"].map(QUAD)

# --- dimensión Actor
for a in ["Actor1", "Actor2"]:
    obt[f"{a}CountryDesc"]   = obt[f"{a}CountryCode"].map(LK["CAMEO.country"])
    obt[f"{a}Type1Desc"]     = obt[f"{a}Type1Code"].map(LK["CAMEO.type"])
    obt[f"{a}KnownGroupDesc"] = obt[f"{a}KnownGroupCode"].map(LK["CAMEO.knowngroup"])
    obt[f"{a}EthnicDesc"]    = obt[f"{a}EthnicCode"].map(LK["CAMEO.ethnic"])
    obt[f"{a}Religion1Desc"] = obt[f"{a}Religion1Code"].map(LK["CAMEO.religion"])

# --- dimensión Geografía (FIPS)
for g in ["Actor1Geo", "Actor2Geo", "ActionGeo"]:
    obt[f"{g}_CountryDesc"] = obt[f"{g}_CountryCode"].map(LK["FIPS.country"])
    obt[f"{g}_TypeDesc"]    = obt[f"{g}_Type"].map(GEOT)

# --- dimensión Tiempo
obt["FechaEvento"]      = pd.to_datetime(obt["SQLDATE"], format="%Y%m%d", errors="coerce")
obt["FechaPublicacion"] = pd.to_datetime(obt["DATEADDED"], format="%Y%m%d%H%M%S", errors="coerce")
obt["DiasRetraso"]      = (obt["FechaPublicacion"].dt.normalize() - obt["FechaEvento"]).dt.days
obt["HoraPublicacion"]  = obt["FechaPublicacion"].dt.hour
obt["DiaSemana"]        = obt["FechaPublicacion"].dt.day_name()

# --- dimensión Fuente
obt["Dominio"] = obt["SOURCEURL"].str.extract(r"https?://(?:www\.)?([^/:]+)", expand=False).str.lower()

print(obt.shape)
obt.head(3)

In [ ]:
obt.to_parquet(OBT_PATH, engine="pyarrow", compression="snappy", index=False)
print(f"OBT guardada en {OBT_PATH} — {OBT_PATH.stat().st_size/1e6:,.1f} MB, {obt.shape[0]:,} filas × {obt.shape[1]} columnas")

In [ ]:
# Control de completitud temporal: eventos publicados por hora
serie = obt.set_index("FechaPublicacion").resample("1h").size()
fig, ax = plt.subplots(figsize=(11, 3))
serie.plot(ax=ax, lw=1)
ax.set_title("Eventos publicados por hora (huecos = intervalos de 15' faltantes)")
ax.set_ylabel("eventos"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 2. Limpieza

### 2.1 Nulos y ceros

Primero se perfila **toda la OBT**: porcentaje de nulos y de ceros por columna. Un cero no siempre es un faltante: hay que interpretarlo según el codebook.

In [ ]:
num_cols = obt.select_dtypes("number").columns
perfil = pd.DataFrame({
    "dtype":   obt.dtypes.astype(str),
    "% nulos": obt.isna().mean() * 100,
    "% ceros": (obt[num_cols] == 0).mean().reindex(obt.columns) * 100,
}).sort_values("% nulos", ascending=False)
perfil.round(1)

In [ ]:
null_island = ((obt["ActionGeo_Lat"] == 0) & (obt["ActionGeo_Long"] == 0)).mean() * 100
print(f"ActionGeo en (0,0) 'Null Island': {null_island:.3f} %")
print(f"GoldsteinScale == 0: {(obt['GoldsteinScale'] == 0).mean()*100:.2f} %  (valor válido: evento neutro)")
print(f"AvgTone == 0:        {(obt['AvgTone'] == 0).mean()*100:.2f} %  (valor válido: tono neutro)")
print(f"DiasRetraso < 0:     {(obt['DiasRetraso'] < 0).mean()*100:.3f} %  (inconsistencia temporal)")

top = perfil["% nulos"].head(25)[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top.values, color="#c0504d")
ax.set_xlabel("% nulos"); ax.set_title("Columnas con más nulos en la OBT")
plt.tight_layout(); plt.show()

**Selección de variables analíticas.** Para la detección de atípicos se trabaja con las variables que describen *qué* pasó, *cuánto* se cubrió, *con qué tono*, *dónde* y *quiénes*:

* **Numéricas:** `GoldsteinScale`, `AvgTone`, `NumMentions`, `NumSources`, `NumArticles`, `DiasRetraso`, `IsRootEvent`, `ActionGeo_Lat`, `ActionGeo_Long`.
* **Categóricas:** `EventRootCode`, `QuadClass`, `Actor1CountryCode`, `Actor2CountryCode`, `ActionGeo_CountryCode`, `Actor1Type1Code`, `ActionGeo_Type`.

Las columnas con nulos masivos (`*KnownGroupCode`, `*EthnicCode`, `*Religion*`, `*Type2/3Code`, geografía de actores) quedan **descartadas** del análisis: superan ampliamente el 50 % de faltantes y su imputación inventaría información. Siguen disponibles en la OBT para interpretar los atípicos.

**Criterios de tratamiento**

| Variable | Situación | Criterio |
|---|---|---|
| `ActionGeo_Lat/Long` | nulos y (0,0) "Null Island" | (0,0) → nulo; imputación por **mediana del país** (`ActionGeo_CountryCode`); si no hay país, se **descarta** la fila (sin ubicación el evento no es localizable) |
| `GoldsteinScale` | nulos esporádicos | imputación por la **moda del `EventCode`**: en CAMEO la escala de Goldstein es función del código de evento |
| `AvgTone` | nulos esporádicos | imputación por **mediana** del `EventRootCode` |
| `NumMentions/Sources/Articles` | ceros o nulos | inconsistentes (todo evento tiene al menos 1 artículo) → **descarte** |
| `DiasRetraso` | nulo o negativo | fecha inválida → **descarte** |
| `GoldsteinScale = 0`, `AvgTone = 0` | ceros | **se conservan**: son valores neutros válidos, no faltantes |
| Categóricas | nulos (p.ej. actor sin país) | categoría explícita **`UNKNOWN`**: el faltante es informativo (MNAR — p.ej. actores no estatales) |

In [ ]:
NUM = ["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles",
       "DiasRetraso", "IsRootEvent", "ActionGeo_Lat", "ActionGeo_Long"]
CAT = ["EventRootCode", "QuadClass", "Actor1CountryCode", "Actor2CountryCode",
       "ActionGeo_CountryCode", "Actor1Type1Code", "ActionGeo_Type"]

df = obt[["GLOBALEVENTID", "EventCode"] + NUM + CAT].copy().set_index("GLOBALEVENTID")
n0 = len(df)
bitacora = []

# (0,0) -> nulo
mask = (df["ActionGeo_Lat"] == 0) & (df["ActionGeo_Long"] == 0)
df.loc[mask, ["ActionGeo_Lat", "ActionGeo_Long"]] = np.nan
bitacora.append(("Null Island -> NaN", int(mask.sum())))

# coordenadas: mediana por país
for c in ["ActionGeo_Lat", "ActionGeo_Long"]:
    falt = df[c].isna().sum()
    df[c] = df[c].fillna(df.groupby("ActionGeo_CountryCode")[c].transform("median"))
    bitacora.append((f"{c} imputada por mediana del país", int(falt - df[c].isna().sum())))
antes = len(df); df = df.dropna(subset=["ActionGeo_Lat", "ActionGeo_Long"])
bitacora.append(("Descartadas sin ubicación", antes - len(df)))

# Goldstein: moda por EventCode
falt = df["GoldsteinScale"].isna().sum()
moda = df.groupby("EventCode")["GoldsteinScale"].agg(lambda s: s.mode().iat[0] if s.notna().any() else np.nan)
df["GoldsteinScale"] = df["GoldsteinScale"].fillna(df["EventCode"].map(moda))
bitacora.append(("GoldsteinScale imputada por moda del EventCode", int(falt - df["GoldsteinScale"].isna().sum())))

# AvgTone: mediana por EventRootCode
falt = df["AvgTone"].isna().sum()
df["AvgTone"] = df["AvgTone"].fillna(df.groupby("EventRootCode")["AvgTone"].transform("median"))
bitacora.append(("AvgTone imputado por mediana del EventRootCode", int(falt - df["AvgTone"].isna().sum())))

# conteos inconsistentes y fechas inválidas
antes = len(df)
cnt = ["NumMentions", "NumSources", "NumArticles"]
df = df[(df[cnt] > 0).all(axis=1) & df["DiasRetraso"].ge(0)]
df = df.dropna(subset=NUM)
bitacora.append(("Descartadas por conteos en 0 / fecha inválida / NaN residual", antes - len(df)))

# categóricas: nulo -> UNKNOWN
df["QuadClass"] = df["QuadClass"].map(QUAD)
df["ActionGeo_Type"] = df["ActionGeo_Type"].map(GEOT)
for c in CAT:
    n = df[c].isna().sum()
    df[c] = df[c].fillna("UNKNOWN").astype(str)
    bitacora.append((f"{c}: nulo -> 'UNKNOWN'", int(n)))

df = df.drop(columns="EventCode")
display(pd.DataFrame(bitacora, columns=["Acción", "Filas afectadas"]))
print(f"Filas: {n0:,} -> {len(df):,}  (descartadas {100*(1-len(df)/n0):.2f} %)")
assert df.isna().sum().sum() == 0

**Transformaciones previas al ACP.** El ACP es sensible a la asimetría y a la escala: las variables de conteo de cobertura mediática tienen colas muy pesadas (pocos eventos acaparan miles de menciones), por lo que se aplica `log1p`. Las coordenadas se pasan a la **esfera unitaria** (x, y, z) para que dos puntos cercanos a ±180° de longitud no queden artificialmente alejados.

In [ ]:
LOG = ["NumMentions", "NumSources", "NumArticles", "DiasRetraso"]
asim_antes = df[LOG].skew()
for c in LOG:
    df[c] = np.log1p(df[c])
asim = pd.DataFrame({"asimetría original": asim_antes, "asimetría log1p": df[LOG].skew()})
display(asim.round(2))

lat, lon = np.radians(df.pop("ActionGeo_Lat")), np.radians(df.pop("ActionGeo_Long"))
df["Geo_x"], df["Geo_y"], df["Geo_z"] = np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)
NUM_F = ["GoldsteinScale", "AvgTone"] + LOG + ["IsRootEvent", "Geo_x", "Geo_y", "Geo_z"]

df[NUM_F].describe().T.round(3)

### 2.2 Reducción de cardinalidad

Las categorías con frecuencia relativa **menor al 5 %** de la muestra se agrupan en `OTHER`. Esto evita cientos de columnas indicadoras casi vacías (p.ej. ~250 códigos de país) que dominarían la varianza del ACP y generarían falsos atípicos por pura rareza de etiqueta.

In [ ]:
def agrupar_marginales(s: pd.Series, umbral: float) -> pd.Series:
    freq = s.value_counts(normalize=True)
    conservar = freq[freq >= umbral].index
    return s.where(s.isin(conservar), "OTHER")

card = []
for c in CAT:
    antes = df[c].nunique()
    df[c] = agrupar_marginales(df[c], UMBRAL_RARO)
    card.append((c, antes, df[c].nunique(), (df[c] == "OTHER").mean() * 100))
display(pd.DataFrame(card, columns=["variable", "cardinalidad original", "cardinalidad reducida", "% en OTHER"])
          .round(1).set_index("variable"))

for c in CAT:
    print(f"\n{c}:", df[c].value_counts(normalize=True).mul(100).round(1).to_dict())

### 2.3 Variables indicadoras (dummies)

Cada categórica se transforma en variables 0/1. Se elimina una **categoría de referencia** por variable (`OTHER` si existe; si no, la más frecuente): con todas las dummies, sus columnas suman 1 y generan colinealidad perfecta (autovalores nulos en el ACP).

In [ ]:
dummies = pd.get_dummies(df[CAT], prefix=CAT, prefix_sep="=", dtype=np.uint8)
referencias = []
for c in CAT:
    ref = f"{c}=OTHER" if f"{c}=OTHER" in dummies else f"{c}={df[c].mode().iat[0]}"
    referencias.append(ref)
dummies = dummies.drop(columns=referencias)
print("Categorías de referencia eliminadas:", referencias)

X = pd.concat([df[NUM_F].astype(np.float32), dummies.astype(np.float32)], axis=1)
print(f"Matriz de características: {X.shape[0]:,} eventos × {X.shape[1]} variables")
X.head()

### 2.4 Reducción de la dimensionalidad: ACP

Se estandarizan todas las características (media 0, desvío 1) para que ninguna domine por su escala, y se ajusta el ACP con todas las componentes. Se comparan cuatro criterios para la cantidad mínima de componentes:

* **Varianza acumulada** ≥ 80 % (criterio adoptado) y ≥ 90 %.
* **Kaiser:** autovalores > 1 (la componente explica más que una variable original estandarizada).
* **Codo** del *scree plot* (máxima distancia a la recta entre el primer y el último autovalor).

In [ ]:
scaler = StandardScaler()
Xs = scaler.fit_transform(X.values).astype(np.float32)

try:
    pca = PCA(svd_solver="covariance_eigh", random_state=SEED).fit(Xs)   # eficiente con n >> p
except (ValueError, TypeError):
    pca = PCA(svd_solver="full", random_state=SEED).fit(Xs)

autoval = pca.explained_variance_
evr = pca.explained_variance_ratio_
cum = evr.cumsum()

def codo(v):
    x = np.arange(len(v)); x1, y1, x2, y2 = 0, v[0], len(v) - 1, v[-1]
    d = np.abs((y2 - y1) * x - (x2 - x1) * v + x2 * y1 - y2 * x1) / np.hypot(y2 - y1, x2 - x1)
    return int(d.argmax()) + 1

criterios = {
    "Varianza acumulada ≥ 80 %": int(np.searchsorted(cum, 0.80) + 1),
    "Varianza acumulada ≥ 90 %": int(np.searchsorted(cum, 0.90) + 1),
    "Kaiser (λ > 1)":            int((autoval > 1).sum()),
    "Codo (scree plot)":         codo(autoval),
}
K = int(np.searchsorted(cum, UMBRAL_VAR) + 1)
display(pd.Series(criterios, name="componentes").to_frame())
print(f"\n==> K = {K} componentes (de {X.shape[1]} variables) explican {cum[K-1]*100:.1f} % de la varianza")

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
idx = np.arange(1, len(autoval) + 1)
ax1.bar(idx, evr * 100, color="#4f81bd", alpha=0.8, label="varianza explicada")
ax1.set_xlabel("componente principal"); ax1.set_ylabel("% varianza explicada")
ax2 = ax1.twinx()
ax2.plot(idx, cum * 100, "o-", color="#c0504d", ms=3, label="acumulada")
ax2.axhline(UMBRAL_VAR * 100, ls="--", color="gray")
ax2.axvline(K, ls=":", color="black")
ax2.set_ylabel("% acumulado"); ax2.set_ylim(0, 102); ax2.grid(False)
ax1.set_title(f"Scree plot y varianza acumulada — K = {K}")
fig.legend(loc="center right", bbox_to_anchor=(0.88, 0.5))
plt.tight_layout(); plt.show()

In [ ]:
# Cargas (loadings) de las primeras componentes: correlación variable-componente
nc = min(K, 8)
loadings = pd.DataFrame(pca.components_[:nc].T * np.sqrt(autoval[:nc]),
                        index=X.columns, columns=[f"CP{i+1}" for i in range(nc)])
fig, ax = plt.subplots(figsize=(1.0 * nc + 3, 0.28 * len(X.columns) + 1))
im = ax.imshow(loadings.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(nc)); ax.set_xticklabels(loadings.columns)
ax.set_yticks(range(len(X.columns))); ax.set_yticklabels(X.columns, fontsize=7)
ax.grid(False); plt.colorbar(im, ax=ax, label="carga")
ax.set_title("Cargas de las variables en las primeras componentes")
plt.tight_layout(); plt.show()

for c in loadings.columns[:4]:
    top = loadings[c].abs().sort_values(ascending=False).head(4).index
    print(f"{c} ({evr[int(c[2:])-1]*100:.1f} %):", ", ".join(f"{v} ({loadings.at[v, c]:+.2f})" for v in top))

**Interpretación.** Las cargas permiten nombrar las componentes: típicamente una componente de **cobertura mediática** (NumMentions/NumSources/NumArticles), una de **conflicto vs. cooperación** (GoldsteinScale, QuadClass, EventRootCode, AvgTone), y componentes **geográficas** (Geo_x/y/z y país del ActionGeo). Al trabajar con una matriz mixta (continuas + dummies), la varianza se reparte entre muchas direcciones, por lo que K suele ser una fracción moderada de las variables originales; aun así el ACP elimina las direcciones redundantes y de ruido.

### 2.5 Detección de eventos atípicos multivariantes

Se siguen las familias de métodos del notebook de referencia de la cátedra (*Cardiotocography outliers*), aplicadas sobre el espacio reducido del ACP (la primera de las "otras ideas" que propone ese notebook):

| Familia | Método | Score | Criterio de marcado |
|---|---|---|---|
| Distancia global | **Z-score** de la distancia euclídea al centroide | $z(d_i)$ | $z > 3$ |
| Distancia global | **Tukey fences** sobre la misma distancia | $d_i$ | $d_i > Q_3 + 2\cdot IQR$ |
| ACP — espacio reducido | **T² de Hotelling** (Mahalanobis en las K componentes) | $T^2_i=\sum_k z_{ik}^2/\lambda_k$ | $\chi^2_{0.999}(K)$ |
| ACP — reconstrucción | **SPE / Q** (error fuera del subespacio) | $\lVert x_i-\hat x_i\rVert^2$ | aproximación de Box |
| Agrupamiento | **K-Means**: distancia al centroide de su grupo | $d(x_i, c_{g(i)})$ | 1 % superior |
| Proximidad | **k-NN**: distancia media a los k vecinos | $\bar d_k$ | 1 % superior |
| Densidad | **LOF** (densidad local relativa) | LOF | 1 % superior |
| Densidad | **DBSCAN**: ruido (lejos de todo punto núcleo) | — | etiqueta de ruido |
| Subespacios | **Isolation Forest** | profundidad media de aislamiento | 1 % superior |
| Red neuronal | **Autoencoder**: error de reconstrucción | MSE por evento | 1 % superior |

**Diferencias respecto del notebook de referencia.**
* El CTG tiene etiquetas (NSP), por eso allí se evalúa con matriz de confusión, *recall* y precisión. **GDELT no tiene etiquetas de "evento raro"**: la validación es por **acuerdo entre métodos** (índice de Jaccard) y **consenso por mayoría de votos**, más la inspección manual de las fuentes.
* El CTG tiene 2.126 registros; aquí hay ~10⁶ eventos. Los métodos de vecindad (k-NN, LOF, DBSCAN) son O(n²): se ajustan sobre una **muestra de referencia** de `N_REF` eventos y se **evalúan sobre todos** (modo *novelty*). K-Means usa la variante *MiniBatch*.
* Los métodos de score continuo usan el mismo corte (`TOP_FRAC` = 1 %, como el K-Means del notebook de referencia) para que los votos sean comparables.

In [ ]:
rng = np.random.default_rng(SEED)
Z = pca.transform(Xs)[:, :K].astype(np.float64)
lam = autoval[:K]
n = len(Z)
idx_ref = rng.choice(n, size=min(n, N_REF), replace=False)       # muestra de referencia para métodos O(n²)
Z_ref = Z[idx_ref]

scores, flags, familia = {}, {}, {}
def registrar(nombre, fam, score, flag):
    scores[nombre], flags[nombre], familia[nombre] = np.asarray(score, float), np.asarray(flag, bool), fam
    print(f"{nombre:<20} {fam:<24} marcados: {flags[nombre].sum():>9,}  ({flags[nombre].mean()*100:.2f} %)")

def top(score, frac=TOP_FRAC):
    return score > np.quantile(score, 1 - frac)

#### 2.5.1 Métodos por distancia global

Distancia euclídea de cada evento al vector promedio (en el espacio de componentes), convertida a Z-score. Antes de usar el corte de 3 desvíos conviene preguntarse **en qué supuesto se apoya**: que las distancias sean normales. Se verifica con curtosis y la prueba de Anderson-Darling (sobre una muestra, ya que con n ≈ 10⁶ cualquier desvío mínimo resulta significativo). Si no hay normalidad, las **Tukey fences** (basadas en cuartiles) son más robustas.

In [ ]:
dist_centro = np.linalg.norm(Z - Z.mean(axis=0), axis=1)
z_dist = zscore(dist_centro)

print(f"Curtosis (exceso) de las distancias: {pd.Series(z_dist).kurtosis():.2f}  (normal = 0)")
ad = anderson(rng.choice(z_dist, size=min(n, 5_000), replace=False), dist="norm")
print(f"Anderson-Darling: estadístico = {ad.statistic:.2f} | valor crítico al 5 % = {ad.critical_values[2]:.3f} -> "
      + ("NO normal" if ad.statistic > ad.critical_values[2] else "compatible con normal"))

q1, q3 = np.percentile(dist_centro, [25, 75])
lim_tukey = q3 + 2 * (q3 - q1)

registrar("Z-score", "Distancia global", z_dist, z_dist > 3)
registrar("Tukey", "Distancia global", dist_centro, dist_centro > lim_tukey)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(z_dist, bins=200, color="#4f81bd", alpha=0.8)
for x, c in [(2, "orange"), (3, "red")]:
    ax.axvline(x, color=c, ls="--", label=f"Z = {x}")
ax.axvline((lim_tukey - dist_centro.mean()) / dist_centro.std(), color="black", ls=":", label="Tukey 2·IQR")
ax.set_yscale("log"); ax.set_xlabel("Z-score de la distancia al centro"); ax.set_ylabel("eventos (log)")
ax.set_title("Distribución de las distancias al centro"); ax.legend()
plt.tight_layout(); plt.show()

#### 2.5.2 Modelo ACP: espacio reducido (T²) y reconstrucción (SPE)

* **T² de Hotelling**: a diferencia de la distancia euclídea anterior, pondera cada componente por su autovalor (distancia de Mahalanobis). Detecta combinaciones **extremas pero coherentes** con la estructura (p.ej. cobertura masiva). Límite $\chi^2_{0.999}(K)$.
* **SPE / Q**: error cuadrático de reconstrucción desde las K componentes. Detecta eventos que **rompen la correlación** habitual (p.ej. tono muy positivo en un conflicto material). Límite de Box $g\,\chi^2_{\alpha}(h)$, con $g=\mathrm{var}(Q)/(2\bar Q)$ y $h=2\bar Q^2/\mathrm{var}(Q)$.

In [ ]:
T2 = (Z**2 / lam).sum(axis=1)
lim_T2 = chi2.ppf(ALPHA, K)

Xc = Xs.astype(np.float64) - pca.mean_
SPE = np.clip((Xc**2).sum(axis=1) - (Z**2).sum(axis=1), 0, None)   # ||x||² = ||z||² + ||e||²
m, v = SPE.mean(), SPE.var()
g, h = v / (2 * m), 2 * m**2 / v
lim_SPE = g * chi2.ppf(ALPHA, h)

print(f"Límite T² = {lim_T2:,.2f} | Límite SPE = {lim_SPE:,.2f}")
registrar("T² Hotelling", "ACP espacio reducido", T2, T2 > lim_T2)
registrar("SPE (Q)", "ACP reconstrucción", SPE, SPE > lim_SPE)

#### 2.5.3 Modelos basados en agrupamiento: K-Means

En vez de medir contra el centro de toda la masa de datos, se agrupan los eventos y se mide la distancia de cada uno **al centroide de su propio grupo**. Un evento lejos de todos los perfiles típicos de cobertura es candidato a atípico.

In [ ]:
kmeans = MiniBatchKMeans(n_clusters=K_CLUSTERS, random_state=SEED, batch_size=8192, n_init=5).fit(Z)
dist_km = np.linalg.norm(Z - kmeans.cluster_centers_[kmeans.labels_], axis=1)

ss_total = ((Z - Z.mean(0))**2).sum()
ss_within = (dist_km**2).sum()
print(f"Varianza explicada por los grupos (SSB/SST): {1 - ss_within/ss_total:.1%}")
print("Tamaño de los grupos:", np.bincount(kmeans.labels_).tolist())
registrar("K-Means", "Agrupamiento", dist_km, top(dist_km))

#### 2.5.4 Modelos basados en proximidad: k-NN

Grado de aislamiento = distancia media a los k vecinos más cercanos. No asume un centroide: un evento es raro si **no tiene vecinos parecidos**, aunque esté cerca del centro global.

In [ ]:
nn = NearestNeighbors(n_neighbors=K_VECINOS + 1).fit(Z_ref)
d_vec, _ = nn.kneighbors(Z)
en_ref = np.zeros(n, dtype=bool); en_ref[idx_ref] = True
# los puntos de la muestra de referencia se encuentran a sí mismos (distancia 0): se descarta ese vecino
knn_score = np.where(en_ref, d_vec[:, 1:].mean(axis=1), d_vec[:, :-1].mean(axis=1))
registrar("k-NN", "Proximidad", knn_score, top(knn_score))

#### 2.5.5 Modelos basados en densidad: LOF y DBSCAN

* **LOF** compara la densidad local de cada evento con la de sus vecinos: detecta atípicos *locales* (raros dentro de su vecindario aunque no lo sean globalmente).
* **DBSCAN** no asume grupos esféricos: los puntos que no alcanzan `min_samples` vecinos dentro de un radio `eps` son **ruido**. El `eps` se elige con el **gráfico de k-distancias** (codo). Como DBSCAN no puede evaluar puntos nuevos, se ajusta sobre la muestra de referencia y luego un evento se considera ruido si su distancia al punto núcleo más cercano supera `eps`.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=K_VECINOS, novelty=True).fit(Z_ref)
lof_score = -lof.score_samples(Z)                       # mayor = menos denso que sus vecinos
registrar("LOF", "Densidad", lof_score, top(lof_score))

# DBSCAN: elección de eps con el gráfico de k-distancias
MIN_SAMPLES = 2 * K
d_k, _ = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(Z_ref).kneighbors(Z_ref)
k_dist = np.sort(d_k[:, -1])
eps = float(k_dist[codo(k_dist) - 1])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(k_dist, lw=1.5)
ax.axhline(eps, color="red", ls="--", label=f"codo: eps ≈ {eps:.2f}")
ax.set_xlabel("eventos de referencia ordenados"); ax.set_ylabel(f"distancia al {MIN_SAMPLES}-ésimo vecino")
ax.set_title("Gráfico de k-distancias para elegir eps"); ax.legend()
plt.tight_layout(); plt.show()

db = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit(Z_ref)
n_grupos = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"DBSCAN en la muestra: {n_grupos} grupos, {np.mean(db.labels_ == -1)*100:.2f} % ruido")

nucleos = Z_ref[db.core_sample_indices_]
d_nucleo = NearestNeighbors(n_neighbors=1).fit(nucleos).kneighbors(Z)[0][:, 0]
registrar("DBSCAN", "Densidad", d_nucleo, d_nucleo > eps)

#### 2.5.6 Modelos basados en subespacios: Isolation Forest

Construye árboles que eligen al azar una variable y un punto de corte; los atípicos quedan **aislados en pocas particiones** (profundidad media baja). No supone normalidad ni usa distancias.

In [ ]:
muestra_if = rng.choice(n, size=min(n, 300_000), replace=False)
iso = IsolationForest(n_estimators=300, max_samples=512, random_state=SEED, n_jobs=-1).fit(Z[muestra_if])
if_score = -iso.score_samples(Z)                        # mayor = más fácil de aislar
registrar("Isolation Forest", "Subespacios", if_score, top(if_score))

#### 2.5.7 Autoencoder: error de reconstrucción

Versión no lineal de la idea del SPE: una red densa comprime las variables escaladas a un **espacio latente** de pocas dimensiones y las reconstruye. Un evento que la red **no puede reconstruir bien** no se parece a lo que aprendió como típico. Como en GDELT no hay etiquetas, se entrena con una muestra de todos los eventos (los atípicos son una fracción ínfima y casi no influyen en el patrón aprendido).

| capa | salida |
|---|---|
| `Linear(p, 32)` + ReLU | 32 |
| `Linear(32, 16)` + ReLU | 16 |
| `Linear(16, DIM_LATENTE)` | espacio latente |
| decoder: camino inverso | p |

In [ ]:
try:
    import torch
    import torch.nn as nn_t
except ImportError:
    torch = None
    print("PyTorch no disponible: se omite el autoencoder (en Colab viene instalado).")

if torch is not None:
    torch.manual_seed(SEED)
    p = Xs.shape[1]

    class Autoencoder(nn_t.Module):
        def __init__(self, dim):
            super().__init__()
            self.encoder = nn_t.Sequential(nn_t.Linear(p, 32), nn_t.ReLU(), nn_t.Linear(32, 16), nn_t.ReLU(),
                                           nn_t.Linear(16, dim))
            self.decoder = nn_t.Sequential(nn_t.Linear(dim, 16), nn_t.ReLU(), nn_t.Linear(16, 32), nn_t.ReLU(),
                                           nn_t.Linear(32, p))
        def forward(self, x):
            return self.decoder(self.encoder(x))

    idx_ae = rng.choice(n, size=min(n, 300_000), replace=False)
    corte = int(0.9 * len(idx_ae))
    X_tr = torch.tensor(Xs[idx_ae[:corte]]); X_va = torch.tensor(Xs[idx_ae[corte:]])

    ae = Autoencoder(DIM_LATENTE)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    perdida = nn_t.MSELoss()
    hist = []
    for epoca in range(EPOCAS_AE):
        ae.train()
        for lote in torch.randperm(len(X_tr)).split(1024):
            opt.zero_grad()
            l = perdida(ae(X_tr[lote]), X_tr[lote])
            l.backward(); opt.step()
        ae.eval()
        with torch.no_grad():
            hist.append((perdida(ae(X_tr[:50_000]), X_tr[:50_000]).item(), perdida(ae(X_va), X_va).item()))

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot([a for a, _ in hist], label="entrenamiento"); ax.plot([b for _, b in hist], label="validación")
    ax.set_xlabel("época"); ax.set_ylabel("MSE"); ax.set_title("Curva de aprendizaje del autoencoder"); ax.legend()
    plt.tight_layout(); plt.show()

    with torch.no_grad():
        error_rec = np.concatenate([((ae(torch.tensor(b)) - torch.tensor(b))**2).mean(1).numpy()
                                    for b in np.array_split(Xs, max(1, n // 100_000))])
    registrar("Autoencoder", "Red neuronal", error_rec, top(error_rec))

#### 2.5.8 Comparación de métodos y consenso

Sin etiquetas no hay *recall* ni precisión. Se compara:
* cuántos eventos marca cada método,
* el **acuerdo entre métodos** (índice de Jaccard: |A∩B| / |A∪B|): métodos de familias distintas que coinciden refuerzan la evidencia,
* el **consenso**: un evento es atípico si lo marca **la mayoría** de los métodos. El ranking final es el promedio de los percentiles de todos los scores.

In [ ]:
nombres = list(flags)
F = pd.DataFrame(flags, index=df.index)
S = pd.DataFrame(scores, index=df.index)

res = pd.concat([S.add_prefix("score_"), F.add_prefix("flag_")], axis=1)
res["votos"] = F.sum(axis=1)
VOTOS_MIN = len(nombres) // 2 + 1
res["score_atipico"] = S.rank(pct=True).mean(axis=1)
res["atipico"] = res["votos"] >= VOTOS_MIN

jac = pd.DataFrame(index=nombres, columns=nombres, dtype=float)
for a in nombres:
    for b in nombres:
        union = (F[a] | F[b]).sum()
        jac.loc[a, b] = (F[a] & F[b]).sum() / union if union else np.nan

resumen = pd.DataFrame({
    "familia": pd.Series(familia),
    "marcados": F.sum(),
    "% marcados": F.mean() * 100,
    "Jaccard con consenso": [(F[m] & res.atipico).sum() / max((F[m] | res.atipico).sum(), 1) for m in nombres],
}, index=nombres)
display(resumen.round(3))

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(jac.values.astype(float), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(nombres))); ax.set_xticklabels(nombres, rotation=45, ha="right")
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax.text(j, i, f"{jac.iat[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if jac.iat[i, j] > 0.5 else "black")
ax.grid(False); plt.colorbar(im, ax=ax, label="Jaccard")
ax.set_title("Acuerdo entre métodos de detección")
plt.tight_layout(); plt.show()

display(res["votos"].value_counts().sort_index().rename("eventos").to_frame())
print(f"\n==> Atípicos de consenso (≥ {VOTOS_MIN} de {len(nombres)} métodos): "
      f"{res.atipico.sum():,} ({res.atipico.mean()*100:.3f} %)")

In [ ]:
# Diagrama de control T² vs SPE y plano CP1–CP2 (muestra + todos los atípicos de consenso)
sel = np.union1d(rng.choice(n, size=min(n, 60_000), replace=False), np.flatnonzero(res["atipico"].values))
r = res.iloc[sel]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
ax.scatter(r["score_T² Hotelling"][~r.atipico], r["score_SPE (Q)"][~r.atipico],
           s=3, alpha=0.25, color="#7f7f7f", label="normal (muestra)")
ax.scatter(r["score_T² Hotelling"][r.atipico], r["score_SPE (Q)"][r.atipico], s=10, color="#c0504d",
           label="atípico (consenso)")
ax.axvline(lim_T2, ls="--", color="black", lw=1); ax.axhline(lim_SPE, ls="--", color="black", lw=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("T² de Hotelling"); ax.set_ylabel("SPE (Q)"); ax.set_title("Diagrama de control multivariante")
ax.legend()

ax = axes[1]
ax.scatter(Z[sel, 0], Z[sel, 1], s=3, alpha=0.25, c=np.where(res["atipico"].values[sel], "#c0504d", "#7f7f7f"))
ax.set_xlabel(f"CP1 ({evr[0]*100:.1f} %)"); ax.set_ylabel(f"CP2 ({evr[1]*100:.1f} %)")
ax.set_title("Eventos en el plano CP1–CP2 (rojo = atípico de consenso)")
plt.tight_layout(); plt.show()

#### Explicación de cada atípico y exportación

Para que la tabla sea accionable por un periodista, cada atípico lleva:
* las 3 variables con mayor **|z-score|** (qué tiene de extremo), y
* las 3 variables con mayor **residuo** de reconstrucción (qué correlación rompe),

junto con **todos los atributos de la OBT** (actores, lugar, descripción CAMEO, URL de la fuente).

In [ ]:
ids = res.index[res["atipico"]]
pos = res.index.get_indexer(ids)
cols = np.array(X.columns)

zs = Xs[pos]
resid = Xc[pos] - Z[pos] @ pca.components_[:K]

def top3(mat):
    idx = np.argsort(-np.abs(mat), axis=1)[:, :3]
    return [", ".join(f"{cols[j]} ({fila[j]:+.1f})" for j in fila_idx) for fila, fila_idx in zip(mat, idx)]

explic = pd.DataFrame({"Top_zscore": top3(zs), "Top_residuo": top3(resid)}, index=ids)

COLS_OBT = ["GLOBALEVENTID", "FechaEvento", "FechaPublicacion", "DiasRetraso",
            "Actor1Name", "Actor1CountryCode", "Actor1CountryDesc", "Actor2Name", "Actor2CountryCode", "Actor2CountryDesc",
            "EventCode", "EventDesc", "EventRootCode", "EventRootDesc", "QuadClassDesc",
            "GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "IsRootEvent",
            "ActionGeo_FullName", "ActionGeo_CountryCode", "ActionGeo_CountryDesc", "ActionGeo_Lat", "ActionGeo_Long",
            "Dominio", "SOURCEURL"]

atipicos = (obt[COLS_OBT].set_index("GLOBALEVENTID").loc[ids]
            .join(res.loc[ids, [c for c in res.columns if c != "atipico"]])
            .join(explic)
            .sort_values(["votos", "score_atipico"], ascending=False)
            .reset_index())
atipicos.insert(1, "ranking", np.arange(1, len(atipicos) + 1))

atipicos.to_parquet(ATI_PATH, engine="pyarrow", compression="snappy", index=False)
atipicos.to_csv(ATI_PATH.with_suffix(".csv"), index=False)
print(f"Tabla de atípicos exportada: {ATI_PATH} y {ATI_PATH.with_suffix('.csv')} — {len(atipicos):,} eventos")
atipicos.head(25)

#### Perfil de los atípicos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

base = obt.set_index("GLOBALEVENTID").loc[df.index]
lift = (atipicos["EventRootDesc"].value_counts(normalize=True) /
        base["EventRootDesc"].value_counts(normalize=True)).dropna().sort_values().tail(12)
axes[0].barh(lift.index, lift.values, color="#4f81bd"); axes[0].axvline(1, color="black", lw=1)
axes[0].set_title("Sobre-representación (lift) por tipo de evento"); axes[0].tick_params(axis="y", labelsize=8)

pais = atipicos["ActionGeo_CountryDesc"].fillna("Sin dato").value_counts().head(12)[::-1]
axes[1].barh(pais.index, pais.values, color="#c0504d"); axes[1].set_title("Atípicos por país del evento")
axes[1].tick_params(axis="y", labelsize=8)

fondo = base.sample(min(len(base), 50_000), random_state=SEED)
axes[2].scatter(fondo["ActionGeo_Long"], fondo["ActionGeo_Lat"], s=1, alpha=0.15, color="#7f7f7f")
axes[2].scatter(atipicos["ActionGeo_Long"], atipicos["ActionGeo_Lat"],
                s=8 + 30 * atipicos["score_atipico"], color="#c0504d", alpha=0.7)
axes[2].set_xlim(-180, 180); axes[2].set_ylim(-90, 90)
axes[2].set_title("Ubicación de los atípicos"); axes[2].set_xlabel("longitud"); axes[2].set_ylabel("latitud")
plt.tight_layout(); plt.show()

comp = pd.DataFrame({
    "todos (mediana)":    base[["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "DiasRetraso"]].median(),
    "atípicos (mediana)": atipicos[["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles", "DiasRetraso"]].median(),
})
display(comp.round(2))

In [ ]:
# Qué métodos marcaron a los atípicos de consenso
display((atipicos[[f"flag_{m}" for m in nombres]].mean() * 100).rename(lambda c: c[5:])
        .sort_values(ascending=False).round(1).rename("% de los atípicos de consenso").to_frame())

# Los 10 eventos más raros, listos para investigar
(atipicos.head(10)[["ranking", "FechaEvento", "Actor1Name", "Actor2Name", "EventDesc", "QuadClassDesc",
                    "GoldsteinScale", "AvgTone", "NumArticles", "ActionGeo_FullName", "Top_zscore", "SOURCEURL"]])

## 3. Conclusiones

* **Ingesta.** Se construyó una OBT de los últimos 7 días a partir de las tablas *Event* de 15 minutos, verificando integridad (MD5) y completitud temporal, y desnormalizando las dimensiones CAMEO/FIPS. El formato Parquet columnar reduce fuertemente el volumen respecto de los TSV y deja la tabla lista para consultas analíticas.
* **Calidad de datos.** Los faltantes de GDELT no son aleatorios: se concentran en atributos de actores no estatales y en la geografía de actores. Se distinguieron ceros estructurales (Goldstein y tono neutros) de ceros que codifican faltantes (Null Island), y se imputó usando conocimiento del dominio (Goldstein es función del código CAMEO).
* **Cardinalidad y dummies.** El umbral del 5 % redujo cientos de categorías a unas pocas por variable, evitando que la rareza de una etiqueta sea, por sí sola, causa de "atipicidad".
* **ACP.** El número mínimo de componentes se fijó con el criterio de 80 % de varianza acumulada y se contrastó con Kaiser y el codo (ver tabla de criterios). Las primeras componentes se interpretan como cobertura mediática, polaridad conflicto/cooperación y ubicación.
* **Atípicos.** Siguiendo el notebook de referencia de la cátedra se compararon 10 métodos de 6 familias. La prueba de Anderson-Darling muestra que las distancias no son normales, por lo que el corte de 3 desvíos es optimista y conviene apoyarse en métodos robustos (Tukey) o no paramétricos. El índice de Jaccard muestra qué métodos capturan el mismo fenómeno (p.ej. distancia global, T² y K-Means) y cuáles aportan una mirada distinta (SPE y autoencoder detectan rupturas de correlación; LOF, atípicos locales). El consenso por mayoría produce una lista corta y rankeada de eventos. Cada uno incluye la explicación de las variables que lo vuelven raro y la URL de la fuente, lo que permite al equipo periodístico priorizar la verificación.
* **Limitaciones.** Los límites χ² suponen normalidad multivariante, que los datos no cumplen (colas pesadas); por eso se usó consenso y ranking en lugar de un umbral único. Un atípico estadístico puede ser un error de codificación automática de GDELT (actor o lugar mal geocodificado) y no un hecho noticioso: la verificación humana sobre `SOURCEURL` es parte del proceso.

In [ ]:
# Resumen numérico de la corrida
print(f"Ventana: {obt['FechaPublicacion'].min()} -> {obt['FechaPublicacion'].max()}")
print(f"Archivos de 15': {len(archivos) - len(fallidos)} / {len(archivos)}")
print(f"OBT: {len(obt):,} eventos × {obt.shape[1]} columnas")
print(f"Matriz analítica: {X.shape[0]:,} × {X.shape[1]}  |  K = {K} componentes ({cum[K-1]*100:.1f} % var.)")
print(f"Atípicos de consenso: {len(atipicos):,} ({len(atipicos)/len(X)*100:.3f} %)")

In [ ]:
# Descarga opcional de los productos (sólo en Colab)
try:
    from google.colab import files
    files.download(str(ATI_PATH.with_suffix(".csv")))
except ImportError:
    print("No se está ejecutando en Colab; los archivos quedan en", DATA_DIR.resolve())

## Referencias

* GDELT Project. *GDELT 2.0 Event Database* — lista de archivos: http://data.gdeltproject.org/gdeltv2/masterfilelist.txt
* GDELT Project. *GDELT Event Codebook V2.0*: https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
* Schrodt, P. A. *CAMEO Conflict and Mediation Event Observations Codebook v1.1b3*: https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf
* Jackson, J. E. & Mudholkar, G. S. (1979). Control procedures for residuals associated with principal component analysis. *Technometrics*, 21(3).
* Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). Isolation Forest. *ICDM*.